In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
df= pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

In [3]:
columns_to_drop= [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Lat Long",
    "Churn Value",
    "Churn Score",
    "CLTV",
    "Churn Reason"
]

In [4]:
df= df.drop(columns= columns_to_drop)

In [5]:
# Remove City too and rebuild the preprocessing pipeline again
df= df.drop(columns=["City"])

In [6]:
df.shape

(7043, 23)

In [7]:
df.columns


Index(['Zip Code', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label'],
      dtype='str')

In [8]:
df.dtypes

Zip Code               int64
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
dtype: object

In [9]:
(df["Total Charges"]== " ").sum()

np.int64(11)

In [10]:
df["Total Charges"]= pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

In [11]:
df[df["Total Charges"].isna()]

,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label
2234,92408,34.084909,-117.258107,Female,No,Yes,No,0,No,No phone service,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
2438,93526,36.869584,-118.189241,Male,No,No,No,0,Yes,No,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
2568,94401,37.590421,-122.306467,Female,No,Yes,No,0,Yes,No,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
2667,95014,37.306612,-122.080621,Male,No,Yes,Yes,0,Yes,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
2856,95569,40.363446,-123.835041,Female,No,Yes,No,0,No,No phone service,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
4331,90029,34.089953,-118.294824,Male,No,Yes,Yes,0,Yes,No,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
4687,92585,33.739412,-117.173334,Male,No,Yes,Yes,0,Yes,Yes,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
5104,95005,37.078873,-122.090386,Female,No,Yes,Yes,0,Yes,No,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5719,91750,34.144703,-117.770299,Male,No,Yes,Yes,0,Yes,No,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6772,90201,33.970343,-118.171368,Female,No,Yes,Yes,0,Yes,Yes,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [12]:
df["Total Charges"]= df["Total Charges"].fillna(0)

In [13]:
df["Total Charges"].isna().sum()

np.int64(0)

In [14]:
X= df.drop(columns="Churn Label")
y= df["Churn Label"]

In [15]:
X.shape

(7043, 22)

In [16]:
y.shape

(7043,)

In [17]:
X.dtypes

Zip Code               int64
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges        float64
dtype: object

In [18]:
categorical_features= X.select_dtypes(include= ["object", "string"]).columns.tolist()
numerical_features= X.select_dtypes(include= ["int64", "float64"]).columns.tolist()

In [19]:
categorical_features

['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method']

In [20]:
numerical_features

['Zip Code',
 'Latitude',
 'Longitude',
 'Tenure Months',
 'Monthly Charges',
 'Total Charges']

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
numerical_pipeline= Pipeline(
    steps= [
        ("scaler", StandardScaler())
    ]
)

In [23]:
categorical_pipeline= Pipeline(
    steps= [
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [24]:
preprocessor= ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= 42,
    stratify= y
)

In [26]:
X_train.shape

(5634, 22)

In [27]:
X_test.shape

(1409, 22)

In [28]:
X_train_processed= preprocessor.fit_transform(X_train)

X_test_processed= preprocessor.transform(X_test)

In [29]:
X_train_processed.shape

(5634, 49)

In [30]:
# Modeling

In [31]:
from sklearn.linear_model import LogisticRegression

model= LogisticRegression(random_state=42)
model.fit(X_train_processed, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solve

In [32]:
y_train_pred= model.predict(X_train_processed)
y_test_pred= model.predict(X_test_processed)

In [33]:
# Model Evaluation

In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [35]:
print("Train Accuracy: ", accuracy_score(y_train, y_train_pred))
print("Test Accuracy: ", accuracy_score(y_test, y_test_pred))

Train Accuracy:  0.8145189918352858
Test Accuracy:  0.7991483321504613


In [36]:
cm= confusion_matrix(y_test, y_test_pred)
print(cm)

[[913 122]
 [161 213]]


In [37]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

          No       0.85      0.88      0.87      1035
         Yes       0.64      0.57      0.60       374

    accuracy                           0.80      1409
   macro avg       0.74      0.73      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [38]:
from sklearn.metrics import roc_auc_score

# Oth_coln: No, 1st_coln: Yes
y_test_prob= model.predict_proba(X_test_processed)[:, 1]

roc_auc= roc_auc_score(y_test, y_test_prob)

print("ROC-AUC: ", roc_auc)

ROC-AUC:  0.8477899196569273


In [39]:
# Threshold Tuning

In [40]:
y_pred_04= np.where(y_test_prob>= 0.4, "Yes", "No")

print(classification_report(y_test, y_pred_04))

              precision    recall  f1-score   support

          No       0.87      0.82      0.85      1035
         Yes       0.58      0.67      0.62       374

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.78      0.79      1409

